# EXP_011 — Image-Only Baseline (ConvNeXt + MSE)
**Phase 1 | Baseline Establishment**
Research question: How much predictive signal exists in review images alone?
- Image model: `convnext_base_in22k` | Text branch: DISABLED | Fusion: none | Loss: MSE | Seed: 42 | AMP: enabled

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

Cloning into 'SE365'...
remote: Enumerating objects: 13294, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 13294 (delta 112), reused 121 (delta 71), pack-reused 13129 (from 1)
Receiving objects: 100% (13294/13294), 873.18 MiB | 15.83 MiB/s, done.
Resolving deltas: 100% (350/350), done.
/content/SE365


### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!gdown --id 11WoeUn2visKtGN5oOX9c2I6Grz3P88vD -O data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=11WoeUn2visKtGN5oOX9c2I6Grz3P88vD
From (redirected): https://drive.google.com/uc?id=11WoeUn2visKtGN5oOX9c2I6Grz3P88vD&confirm=t&uuid=dcf6543f-6c01-411a-922e-dbe1bc6c39eb
To: /content/SE365/data.zip
100% 4.02G/4.02G [00:47<00:00, 84.5MB/s]
total 1344
drwxr-xr-x  4 root root    4096 Jun 16 09:21 .
drwxr-xr-x 11 root root    4096 Jun 23 05:13 ..
drwxr-xr-x  2 root root 1359872 Jun 16 09:59 image
drwxr-xr-x  2 root root    4096 Jun 16 09:21 text


### STEP 4: Configure paths

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_011_image_only_convnext_meanpool_mse'
DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts will be saved to: {DRIVE_EXP_PATH}')

Artifacts will be saved to: /content/drive/MyDrive/SE365/experiments/EXP_011_image_only_convnext_meanpool_mse


### STEP 5: Train

In [ ]:
!python main.py \
  --mode train_image \
  --image_model_name convnext_base_in22k \
  --epochs 20 \
  --batch_size 32 \
  --lr 2e-5 \
  --grad_accum_steps 1 \
  --patience 5 \
  --loss_fn mse \
  --seed 42 \
  --use_amp \
  --exp_id EXP_011_image_only_convnext_meanpool_mse \
  --exp_dir ./experiments

====== MODE: TRAIN_IMAGE ======
Using device: cuda
Seed: 42 | Experiment: EXP_011_image_only_convnext_meanpool_mse
config.json: 100% 615/615 [00:00<00:00, 1.46MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 106kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:01<00:00, 4.67MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:01<00:00, 7.06MB/s]
preprocessor_config.json: 100% 368/368 [00:00<00:00, 1.98MB/s]
/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(
model.safetensors: 100% 440M/440M [00:07<00:00, 62.1MB/s]
/content/SE365/Trainer.py:82: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.use_amp)
2026-06-23 05:16:04 ====== START: 2026-06-23T05:16:04.829577 ======
2026-06-23 05:16:04 Experiment: EXP_011_image_only_co

### STEP 6: Save to Drive + print metrics

In [ ]:
import json
!cp -r ./experiments/$EXP_ID/* $DRIVE_EXP_PATH/

with open(f'./experiments/{EXP_ID}/metrics.json') as f:
    m = json.load(f)

print(f'\n=== {EXP_ID} Results ===')
print(f"Loss (val)   : {m['loss']:.4f}")
print()
print("             MAE      RMSE      R2")
print(f"  food     : {m['mae_food']:.4f}   {m['rmse_food']:.4f}   {m['r2_food']:.4f}")
print(f"  price    : {m['mae_price']:.4f}   {m['rmse_price']:.4f}   {m['r2_price']:.4f}")
print(f"  atmos    : {m['mae_atmos']:.4f}   {m['rmse_atmos']:.4f}   {m['r2_atmos']:.4f}")
print(f"  service  : {m['mae_service']:.4f}   {m['rmse_service']:.4f}   {m['r2_service']:.4f}")
print(f"  overall  : {m['mae_overall']:.4f}   {m['rmse_overall']:.4f}   {m['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {m['mean_mae']:.4f}")
print(f"  aspect_mae : {m['aspect_mae']:.4f}")
print(f"  overall_mae: {m['overall_mae']:.4f}")


=== EXP_011_image_only_convnext_meanpool_mse Results ===
Loss (val)   : 4.3926

             MAE      RMSE      R2
  food     : 1.6013   2.2630   0.0270
  price    : 1.4951   2.0552   0.0543
  atmos    : 1.4166   1.9152   0.0549
  service  : 1.5809   2.2231   0.0361
  overall  : 1.3808   1.9642   0.0525

  mean_mae   : 1.4949
  aspect_mae : 1.5235
  overall_mae: 1.3808
